<a href="https://colab.research.google.com/github/BreckEmert/repurposed-tokens-experiment/blob/main/LoRA_sanity_run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ──────────────────────────────────────────────────────────────
# Headers are in the code because they look nice
# "If my commit messages don't have emojis, how would you know how I feel?"
# - ProgrammersAreAlsoHuman, '0.1x engineers'
# ──────────────────────────────────────────────────────────────

"""This code trains several models to perform a 3SUM task generated
by https://github.com/JacobPfau/fillerTokens/tree/master, comparing
model training curves, attention maps, and test set performance in
order to highlight the role of filler tokens."""

# I don't think we need these:
# !pip uninstall -y torch torchvision torchaudio
# !pip install -U bitsandbytes accelerate transformers peft

# Try/except to avoid repeated install; not sure of a better way
try:
    import bitsandbytes as bnb
except:
    !pip install --prefer-binary bitsandbytes

    !pip uninstall -y datasets fsspec gcsfs
    !pip install -U "datasets>=2.19.1" "fsspec>=2024.3.1"


In [2]:
# ──────────────────────────────────────────────────────────────
# Imports
# ──────────────────────────────────────────────────────────────
import datetime as dt
import itertools
import json
import os
import tempfile
import pathlib
import shutil
import re
import textwrap
from pathlib import Path
from glob import glob
from functools import partial
from typing import Final
from types import SimpleNamespace as ns
from collections import Counter

import numpy as np
import pandas as pd
import polars as pl
import torch
from torch.nn.utils.rnn import pad_sequence
from datasets import load_dataset, load_from_disk
from google.colab import drive
from huggingface_hub import notebook_login
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


In [3]:
# ──────────────────────────────────────────────────────────────
# Global variables and namespace for the run
# ──────────────────────────────────────────────────────────────
# Dataset settings
MATCH_LENGTH = 12
DIMENSION = 3

# Run namespace
presets = {
    'direct': ns(
        name='direct',
        filler_tok=None,
        generate_raw=True,
        cot_rate=0,
        no_filler_rate=1
    ),
    'cot': ns(
        name='cot',
        filler_tok=None,
        generate_raw=True,
        cot_rate=1,
        no_filler_rate=1
    ),
    'dot': ns(
        name='dot',
        filler_tok=' .',
        generate_raw=True,
        cot_rate=0,
        no_filler_rate=0
    ),
    'low_S': ns(
        name='low_S',
        filler_tok='big',
        generate_raw=False
    ),
    'high_S': ns(
        name='high_S',
        filler_tok='neu',
        generate_raw=False
    )
}
RUN = presets['direct']

# Output paths
BASE_DIR = "/content/drive/MyDrive/Colab_Files/repurposed_tokens"
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"  # or 3-8B
MODEL_NAME = MODEL_ID.split('/')[-1]
RUN.dir = os.path.join(BASE_DIR, MODEL_NAME, RUN.name)
MODEL_DIR = os.path.join(RUN.dir, 'model')
os.makedirs(MODEL_DIR, exist_ok=True)


In [4]:
# ──────────────────────────────────────────────────────────────
# Github and Drive setup
# ──────────────────────────────────────────────────────────────
%cd /content

# Mount drive for data
drive.flush_and_unmount()
if os.path.exists('/content/drive') and not os.path.islink('/content/drive'):
    shutil.rmtree('/content/drive')
drive.mount('/content/drive', force_remount=True)

# Filepaths
FILLER_DIR = "/content/fillerTokens"
ENTROPIX_DIR = "/content/entropix"

# Clone and setup repos
if not os.path.exists(FILLER_DIR):
    !git clone https://github.com/BreckEmert/fillerTokens.git
    !pip install -r {FILLER_DIR}/requirements.txt

if not os.path.exists(ENTROPIX_DIR):
    !git clone https://github.com/BreckEmert/entropix.git

# Create symlink to data
if os.path.islink('./data') or os.path.exists('./data'):
    os.unlink('./data')
os.symlink(RUN.dir, './data')

# Set HF cache
os.environ['HF_HOME'] = '/content/drive/MyDrive/HF_cache'


/content
Mounted at /content/drive


In [5]:
# ──────────────────────────────────────────────────────────────
# HF setup
# ──────────────────────────────────────────────────────────────
notebook_login()


# 3SUM Dataset Generation Script

In [6]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model.  Not very related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to highlight the effect of filler tokens without the large amount of runs that are theoretically all relevant here, as we step through runs 2-4.


Outline of data pipeline for clarity:
1) We generate three kinds of samples, direct, CoT and punct:
    533 569 530 814 A False     VERIFY THIS DOESNT HAVE P AND A IM NOT SURE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    371 578 006 519 P 1- 4 0- 7 3- 7- 4- 4 5- 6 A True
    873 545 827 245 P . . . . . . . . . . . . . A False
    # Note that this is prompt followed by answer in the same string
    # Answers are one word long (True/False), so
      future [-1], len(prompt)-1, are referring to this
2) Turn that into a HF dataset which has two columns:
    a) input_ids: tokenized full prompts
    b) labels: same except -100-masked everywhere but answer and eos tokens
3) Lastly, the collator recieves a list of those dicts and
    a) pads batches to the longest of that batch
    b) stacks examples into tensors
    c) leaves labels alone
"""

# ──────────────────────────────────────────────────────────────
# Generate raw math data with scripts.data_match3
# ──────────────────────────────────────────────────────────────
# This must be run because the script is a module:
%cd {FILLER_DIR}

# Generate the datasets if they don't exist
for name, settings in presets.items():
    if not settings.generate_raw:
        continue

    # Get output paths and see if they exist
    train_path = f"{BASE_DIR}/demo3s_trainset.csv"
    test_path = f"{BASE_DIR}/demo3s_testset.csv"
    cfg_path = f"{BASE_DIR}/demo3s_config.json"

    if bool(glob(train_path)) and bool(glob(test_path)):
        continue
    else:
        raise ValueError()  # While you're figuring out filepathing

    # Run the data generation script
    !python -m scripts.data_match3 \
        --name demo3s \
        --length {MATCH_LENGTH} \
        --dimension {DIMENSION} \
        --train_samples 500000 \
        --test_samples 10000 \
        --cot_rate {settings['cot_rate']} \
        --no_filler_rate {settings['no_filler_rate']} \
        --corruption_rate 1.33
    # Args:
        # --length 12  # number of tuples per instance
        # --dimension 3  # number of integers per tuple
        # --cot_rate 0.5  # fraction to annotate with CoT traces
        # --no_filler_rate 0  # 0 so every non-CoT is punctuation-filler
        #  --corruption_rate 1.33  # False instances have avg 1.33 digits randomly replaced (False are generated from True examples and made False by replacing digits)
    # Generates:
        # args_demo3s_YYYY-MM-DD.json  # metadata
        # demo3s_trainset_YYYY-MM-DD.csv
        # demo3s_testset_YYYY-MM-DD.csv

    # Remove the date from their names
    raw_train_csv = glob(f"{BASE_DIR}/demo3s_trainset_*.csv")[0]
    raw_test_csv = glob(f"{BASE_DIR}/demo3s_testset_*.csv")[0]
    raw_cfg_json = glob(f"{BASE_DIR}/args_demo3s_*.json")[0]

    Path(raw_train_csv).rename(train_path)
    Path(raw_test_csv).rename(test_path)
    Path(raw_cfg_json).rename(cfg_path)

    # Saving them permanently into Drive
    for file in glob("data/demo3s_*"):
        shutil.copy(file, BASE_DIR)

# Grab the generated datasets
raw_train_csv = os.path.join(BASE_DIR, "demo3s_trainset.csv")
raw_test_csv = os.path.join(BASE_DIR, "demo3s_testset.csv")
cfg_json = os.path.join(BASE_DIR, "demo3s_config.json")

print(f"Train csv: {pl.scan_csv(raw_train_csv).describe()}\n")
print(f"Test csv: {pl.scan_csv(raw_test_csv).describe()}\n")
print(f"Config json: {open(cfg_json).read()}")

!ls {BASE_DIR}


/content/fillerTokens
Train csv: shape: (9, 2)
┌────────────┬─────────────────────────────────┐
│ statistic  ┆  873 545 827 245 355 421 893 4… │
│ ---        ┆ ---                             │
│ str        ┆ str                             │
╞════════════╪═════════════════════════════════╡
│ count      ┆ 499999                          │
│ null_count ┆ 0                               │
│ mean       ┆ null                            │
│ std        ┆ null                            │
│ min        ┆  000 002 965 813 813 086 565 3… │
│ 25%        ┆ null                            │
│ 50%        ┆ null                            │
│ 75%        ┆ null                            │
│ max        ┆  999 999 999 973 351 326 268 6… │
└────────────┴─────────────────────────────────┘

Test csv: shape: (9, 2)
┌────────────┬─────────────────────────────────┐
│ statistic  ┆  011 807 039 550 951 054 975 6… │
│ ---        ┆ ---                             │
│ str        ┆ str                            

In [7]:
# ──────────────────────────────────────────────────────────────
# Place local datasets in each RUN type
# ──────────────────────────────────────────────────────────────
if RUN.name in ["direct", "dot", "cot"]:
    # Paths to the copied datasets in RUN.dir
    train_path = os.path.join(RUN.dir, f"{RUN.name}_trainset.csv")
    test_path = os.path.join(RUN.dir, f"{RUN.name}_testset.csv")
    cfg_path = os.path.join(RUN.dir, f"{RUN.name}_config.json")

    # Copy the raw data to the respective dir
    shutil.copy(raw_train_csv, train_path)
    shutil.copy(raw_test_csv, test_path)
    shutil.copy(cfg_json, cfg_path)
    print(f"Copied base files to {RUN.dir}")


# ──────────────────────────────────────────────────────────────
# Generate low-S and high-S datasets as well
# ──────────────────────────────────────────────────────────────
def replace_filler(prompt: str, filler_token: str) -> str:
    # Get segments of the example
    question, right = prompt.split(" P", 1)
    filler, answer = prompt.split(" A", 1)

    # Make the new filler
    n_tokens = len(filler.split())
    new_filler = " ".join(filler_token for _ in range(n_tokens))

    # Return the original q/a with the new filler
    return f"{question} P {new_filler} A {answer}"  # AGAIN, VERIFY SPACES

if RUN.name in ["low_S", "high_S"]:
    # Paths to the datasets with replaced-filler in RUN.dir
    train_path = os.path.join(RUN.dir, f"{RUN.name}_trainset.csv")
    test_path = os.path.join(RUN.dir, f"{RUN.name}_testset.csv")

    for raw_path, out_path in [
        (raw_train_csv, train_path),
        (raw_test_csv, test_path)
    ]:
        # Don't regenerate it if it exists
        if os.path.exists(out_path):
            continue

        # Run replace_filler
        df = (
            pl.scan_csv(raw_path)
            .with_columns(
                pl.col('prompt')
                .map_elements(lambda s: replace_filler(s, RUN.filler_tok))
            )
        )

        # Write to RUN.dir
        df.sink_csv(out_path)
        print(f"{RUN.name} dataset generated at {out_path}")
        print(df.head())


Copied base files to /content/drive/MyDrive/Colab_Files/repurposed_tokens/Llama-3.2-1B-Instruct/direct


# Tokenization and HF Dataset Creation

In [8]:
# ──────────────────────────────────────────────────────────────
# Prepare tokenizer
# ──────────────────────────────────────────────────────────────
# Tokenise and make a loss mask
    # We only want loss on the answer token and eos, because we want it to use the periods how it wants, not generate them.
tok = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
tok.pad_token = tok.eos_token

    # Now we can set the ids of important constants
if RUN.filler_tok:
    filler_ids = tok(RUN.filler_tok, add_special_tokens=False)['input_ids']
    RUN.filler_id = torch.tensor(filler_ids).unique()

COT_BEGIN_ID = tok(" P", add_special_tokens=False)['input_ids']
TRUE_ID = tok(" True", add_special_tokens=False)['input_ids']
FALSE_ID = tok(" False", add_special_tokens=False)['input_ids']
assert len(COT_BEGIN_ID + TRUE_ID + FALSE_ID) == 3

print("COT_BEGIN_ID: ", COT_BEGIN_ID)
print("TRUE_ID: ", TRUE_ID)
print("FALSE_ID: ", FALSE_ID)


# ──────────────────────────────────────────────────────────────
# Prepare raw math data into a HF dataset
# ──────────────────────────────────────────────────────────────
# Paths to the tokenized dir
train_tok_dir = os.path.join(RUN.dir, f"{RUN.name}_trainset_tokenized")
test_tok_dir = os.path.join(RUN.dir, f"{RUN.name}_testset_tokenized")

# Only tokenize the raw csvs if they haven't been made
if os.path.exists(train_tok_dir) and os.path.exists(test_tok_dir):
    train_ds = load_from_disk(train_tok_dir)
    test_ds = load_from_disk(test_tok_dir)
else:
    # Grab the data
    datasets = load_dataset(
        'csv',
        data_files={'train': train_path, 'test': test_path},
        header=None,
        column_names=['prompt']
    )
    train_raw = datasets['train']
    test_raw = datasets['test']

    def tokenize_and_mask(batch):
        # Tokenize the batch
        ids_batch = tok(batch['prompt'], add_special_tokens=True)['input_ids']

        # Mask off everything but answer and EOS
        labels = []
        for ids in ids_batch:
            label = [-100]*len(ids)
            # If we wanted the model to produce CoT, we could do [MATCH_LENGTH:] mask instead, but I don't think that helps the research question.
            label[-2:] = ids[-2:]
            labels.append(label)

        return {'input_ids': ids_batch, 'labels': labels}

    train_ds = train_raw.map(tokenize_and_mask, batched=True,
                             remove_columns=['prompt'])
    test_ds = test_raw.map(tokenize_and_mask, batched=True,
                           remove_columns=['prompt'])  # COMPLETELY UNSURE ON THIS REMOVE_COL NAME, CHECK IT

    # Save to Drive for next time
    train_ds.save_to_disk(train_tok_dir)
    test_ds.save_to_disk(test_tok_dir)

# Collation is done in the finetune step, which just pads


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


COT_BEGIN_ID:  [393]
TRUE_ID:  [3082]
FALSE_ID:  [3641]


# Finetune

In [9]:
"""
I plan to use these training runs:
    1) CoT as filler (filler unmasked)
    2) Dots as filler (filler masked out)
    3) Low entropy tokens as filler (filler masked out)
    4) High entropy tokens as filler (filler masked out)
    5) Direct-to-answer

This allows for
    1) A high-quality gold standard for the model.  Not very
        related to my experiment but a super-baseline.
    2) Progressively more difficult tasks to highlight the
        effect of filler tokens without the large amount of
        runs that are theoretically all relevant here, as
        we step through runs 2-4.
"""
# ──────────────────────────────────────────────────────────────
# 3. Attach LoRA adapter to the model
# ──────────────────────────────────────────────────────────────
# Bnb config
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,  # Run this on T4s
)

# Load model and resize embeddings
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype='auto',
    device_map='auto',
    # attn_implementation='flash_attention_2',  # Run this on A100, not T4
)
base_model.resize_token_embeddings(len(tok))

# LoRA config
lora_cfg = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
        # CONFIRM model.print_trainable_parameters() and verify non-zero count - if zero us full names
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

# Now that the model is made we can move this to the gpu
if RUN.filler_tok:
    RUN.filler_id.to(model.device)


trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [ ]:
# ──────────────────────────────────────────────────────────────
# 4. Finetune
# ──────────────────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=os.path.join(MODEL_DIR, 'checkpoints'),
    label_names=['labels'],
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,  # 8*4 = 32 examples per optimizer step
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    max_steps=3_000,
    warmup_steps=200,
    bf16=False,
    fp16=True,
    fp16_full_eval=True,
    optim='paged_adamw_8bit',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',  # DISABLE WANDB FOR QUICK RUNS
)

# Just pad/stacks
    # DCWP defaults to -100 for padding, so we're good there.
    # HF documentation is unacceptable and trying to get their stuff to work is proving to be the worst use of time ever so I'm removing it
# data_collator = DataCollatorWithPadding(
#     tokenizer=tok,
#     pad_to_multiple_of=8,  # Only useful on A100
#     return_tensors='pt',  # I'm from tf so I'll be typing this :D
# )
def custom_data_collator(examples):
    # Convert to tensors
    input_ids = [torch.tensor(ex['input_ids'], dtype=torch.long)
                 for ex in examples]
    labels = [torch.tensor(ex['labels'], dtype=torch.long)
              for ex in examples]

    # Pad to same length
    input_ids = pad_sequence(input_ids, batch_first=True,
                             padding_value=tok.pad_token_id)
    labels = pad_sequence(labels, batch_first=True,
                          padding_value=tok.pad_token_id)

    return {
        'input_ids': input_ids,
        'labels': labels,
        'attention_mask': torch.ones_like(input_ids)
    }

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=custom_data_collator,
)

# Train
trainer.train()
model.save_pretrained(MODEL_DIR)
tok.save_pretrained(MODEL_DIR)


Epoch,Training Loss,Validation Loss


In [ ]:
# Grab a mini-batch from your tokenized dataset
batch = [train_ds[i] for i in range(4)]  # You can adjust the 4 to any small batch size

# Collate (pad) the batch using your DataCollatorWithPadding
collated = data_collator(batch)

# Display shapes and values for inspection
print("input_ids shape: ", collated['input_ids'].shape)
print("labels shape:    ", collated['labels'].shape)
print("attention_mask shape:", collated['attention_mask'].shape)

# For deeper debugging, show the actual padded arrays:
print("\ninput_ids:\n", collated['input_ids'])
print("\nlabels:\n", collated['labels'])
print("\nattention_mask:\n", collated['attention_mask'])


# Evaluation and Visualization

In [ ]:
# ──────────────────────────────────────────────────────────────
# Load the finetuned model and fuse in the LoRA adapter
# ──────────────────────────────────────────────────────────────
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    torch_dtype='auto',
    device_map='auto',
    # attn_implementation='flash_attention_2',  # Run this on A100, not T4
)

# Finetuned LoRA
model = PeftModel.from_pretrained(
    base_model,
    MODEL_DIR,
    is_trainable=False,
    torch_dtype='auto',
    device_map='auto',
)

# Merge
model = model.merge_and_unload()
model.eval()


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention hook
# ──────────────────────────────────────────────────────────────
def _record_attn(layer_idx, module, input, output):
    """Writes (B, H) into CURRENT_FILL (B, L, H)"""
    attention_probs = output[1]  # (B, H, Q, K)  # May throw error - assumes (context, attn_probs, ...) but could be (attn_output, none) etc

    filler_mask = torch.isin(
        TOKEN_IDS_THIS_BATCH.to(model.device),
        RUN.filler_id.to(model.device)
    )[:, None, None, :]  # (B, 1, 1, K)
        # B matches
        # 1 broadcasts across all heads
        # 1 broadcasts across all query positions
        # K matches

    # Compute mean over query dimension
    fill = (attention_probs * filler_mask).sum(-1).mean(-1)  # (B, H)
    CURRENT_FILL[:, layer_idx, :] = fill.cpu()

# Attach to every layer
def attach_attention_hooks(model):
    return [
        block.self_attn.register_forward_hook(partial(_record_attn, layer_idx)) for layer_idx, block in enumerate(model.model.layers)
    ]

# This attaches the hooks, no further work needed
B = input_ids.shape[0]
L = len(model.model.layers)
H = model.config.num_attention_heads
CURRENT_FILL = torch.zeros(B, L, H)  # (B, L, H)

handles = attach_attention_hooks(model)


In [ ]:
# ──────────────────────────────────────────────────────────────
# Attention logging and visualization
# ──────────────────────────────────────────────────────────────
def get_raw_attention(dataloader):
    ATTN_BUCKET.clear()
    L = model.config.num_hidden_layers
    H = model.config.num_attention_heads

    with torch.no_grad():
        for batch in dataloader:
            global TOKEN_IDS_THIS_BATCH, CURRENT_FILL
            TOKEN_IDS_THIS_BATCH = batch["input_ids"]  # (B, S)

            B = TOKEN_IDS_THIS_BATCH.size(0)
            CURRENT_FILL = torch.zeros(B, L, H)  # (B, L, H)  # IF THIS THROWS A TYPE ERROR SET dtype=model.dtype not sure
            _ = model(
                **{k: v.to("cuda") for k, v in batch.items()},
                use_cache=False,
                output_attentions=True
            )
            ATTN_BUCKET.append(CURRENT_FILL)

    raw_attn = torch.cat(ATTN_BUCKET, dim=0)  # (N, L, H)
    return raw_attn

def calculate_enrichment(raw_attn):
    """Adjusts the attention for the percentage of filler tokens
    in the ds otherwise the attention mass is biased
    """
    # Calculate the filler rate
    n_fill, n_total = 0, 0
    for ex in test_ds:
        ids = ex["input_ids"]
        n_total += len(ids)
        n_fill += torch.isin(torch.tensor(ids), RUN.filler_id).sum().item()

    fill_rate = n_fill / n_total if n_total else 0.0
    print(f"Corpus filler rate: {fill_rate:.3%}")
        # THIS MIGHT BE WRONG FOR DIRECT?  BCAUSE THERES NO FILLER TO DILUTE  not sure what I'm looking for with direct

    # Calculate the enrichment
    prob_to_fill = raw_attn.mean(dim=0)  # (L, H)
    return prob_to_fill / max(fill_rate, 1e-9)

def plot_enrichment(mat, title=''):
    """Quick imshow heat-map."""
    import matplotlib.pyplot as plt
    plt.imshow(mat.numpy(), aspect='auto')
    plt.xlabel("Head")
    plt.ylabel("Layer  (0 = bottom)")
    plt.title(title)
    plt.colorbar(label="× corpus filler rate")
    plt.show()

# Get raw attention
raw_attn = get_raw_attention(
    torch.utils.data.DataLoader(
        test_ds,
        batch_size=8,
        shuffle=False,
        collate_fn=data_collator,
    )
)

# Convert to enrichment
enr = calculate_enrichment(raw_attn)

# Visualise & save
plot_enrichment(enr, f"Enrichment - {RUN.name}")
np.save(os.path.join(RUN.dir, "enrichment.npy"), enr.numpy())


In [ ]:
# ──────────────────────────────────────────────────────────────
# Accuracy Eval
# ──────────────────────────────────────────────────────────────
# Remove these if they're still on
for handle in handles:
    handle.remove()

def accuracy(model) -> float:
    """Evaluates model accuracy on test set"""
    # Iterate over test set
    invalid, correct = Counter(), 0
    with torch.no_grad():
        for ex in tqdm(test_ds, leave=False):
            # Forward the input to get an output
            ids = torch.tensor(ex['input_ids']).unsqueeze(0).to('cuda')
            mask = torch.ones_like(ids)
            out = model.generate(
                input_ids=ids,
                attention_mask=mask,
                max_new_tokens=1,
                do_sample=False,
            )

            # Check answer
            pred_id = out[0, -1].item()
            exp_id = ids[0, -1].item()

            if pred_id == exp_id:
                correct += 1
            elif pred not in [TRUE_ID, FALSE_ID]:
                invalid[pred] += 1

    accuracy = correct / len(test_ds)
    return accuracy, invalid

# this is off now that I don't have dots/no dots since that would be handled by aggregating several runs of data, not something we do all at once necessarily?  I guess we can do it all at once after all the models are trained, but it seems useful to have some sort of utility function that works as we go so we can tune the runs.
acc_dots, invalid_dots = accuracy(model)

print(f"Invalid count: {sum(invalid_dots)}")
print(f"Accuracy : {acc_dots:.3f}")
